In [3]:
import glob, os
import pandas as pd

In [4]:
files = sorted(glob.glob('../data/bmrcl/**/*.*', recursive=True))
files = [f for f in files
         if f.lower().endswith(('.csv', '.parquet', '.xlsx')) and '.git' + os.sep not in f]
print('Data files found')

Data files found


In [5]:
for f in files:
    print(f' {f} ({os.path.getsize(f) / 1e6:.1f} MB)')

for f in files:
    if f.lower().endswith('.xlsx'):
        continue
    print('-' * 40)
    print(f)
    try:
        df = pd.read_parquet(f) if f.lower().endswith('.parquet') else pd.read_csv(f, sep=None, engine='python', nrows=200000)
    except Exception as e:
        print(' Could not read:', e)
        continue
    print('Shape: ', df.shape)
    print(df.dtypes)
    print(df.head().to_string())

 ../data/bmrcl\data\station-hourly-exits.parquet (0.1 MB)
 ../data/bmrcl\data\station-hourly.parquet (0.1 MB)
 ../data/bmrcl\data\stationpair-hourly-enhanced.parquet (5.8 MB)
 ../data/bmrcl\data\stationpair-hourly.parquet (2.1 MB)
 ../data/bmrcl\raw\august\station-hourly.xlsx (0.2 MB)
 ../data/bmrcl\raw\august\stationpair-hourly.xlsx (12.6 MB)
 ../data/bmrcl\raw\september\station-hourly.xlsx (0.7 MB)
 ../data/bmrcl\raw\station-codes.csv (0.0 MB)
 ../data/bmrcl\raw\station-names.csv (0.0 MB)
----------------------------------------
../data/bmrcl\data\station-hourly-exits.parquet
Shape:  (95616, 4)
Date           str
Hour         int64
Station        str
Ridership    int64
dtype: object
         Date  Hour    Station  Ridership
0  2025-08-01     0  Attiguppe          0
1  2025-08-01     1  Attiguppe          0
2  2025-08-01     2  Attiguppe          0
3  2025-08-01     3  Attiguppe          0
4  2025-08-01     4  Attiguppe          0
----------------------------------------
../data/bmrcl

In [6]:
import pandas as pd

D = '../data/bmrcl/data/'
ent = pd.read_parquet(D + 'station-hourly.parquet')
ext = pd.read_parquet(D + 'station-hourly-exits.parquet')

for name, df in [('ENTRIES (station-hourly)', ent), ('EXITS' ,ext)]:
    print('-' * 40)
    print(name)
    print('Shape: ', df.shape)
    print('Columns: ', list(df.columns))
    print("dates:", df["Date"].min(), "to", df["Date"].max(), "| unique days:", df["Date"].nunique())
    print("days per month:", df.groupby(df["Date"].str[:7])["Date"].nunique().to_dict())
    print("stations:", df["Station"].nunique())
    print("hours present:", sorted(df["Hour"].unique()))
    print("nulls:", df.isna().sum().to_dict())
    print("total ridership:", df["Ridership"].sum())
    print("rows per station-day (expect 24):", df.groupby(["Station", "Date"]).size().value_counts().to_dict())

print('-' * 50)
print('Top 10 stations by total entries: ')
print(ent.groupby('Station')['Ridership'].sum().sort_values(ascending=False).head(10))

----------------------------------------
ENTRIES (station-hourly)
Shape:  (92280, 4)
Columns:  ['Date', 'Hour', 'Station', 'Ridership']
dates: 2025-08-01 to 2025-09-30 | unique days: 48
days per month: {'2025-08': 18, '2025-09': 30}
stations: 83
hours present: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23)]
nulls: {'Date': 0, 'Hour': 0, 'Station': 0, 'Ridership': 0}
total ridership: 33837882
rows per station-day (expect 24): {24: 3845}
----------------------------------------
EXITS
Shape:  (95616, 4)
Columns:  ['Date', 'Hour', 'Station', 'Ridership']
dates: 2025-08-01 to 2025-09-30 | unique days: 48
days per month: {'2025-08': 18, '2025-09': 30}
stations: 83
hours present: [np.int64(0), np.int64(1), np.int64(2),

In [7]:
all_stations = sorted(ext['Station'].unique())
all_days = sorted(ext['Date'].unique())

full = pd.MultiIndex.from_product([all_stations, all_days], names=['Station', 'Date'])
have = pd.MultiIndex.from_frame(ent[['Station', 'Date']].drop_duplicates())
missing = pd.DataFrame(list(full.difference(have)), columns=['Station', 'Date'])

print('Missing Station-days', len(missing))
print('By station (top 15):')
print(missing.groupby('Station').size().sort_values(ascending=False).head(15))
print('By date (top 10):')
print(missing.groupby('Date').size().sort_values(ascending=False).head(10))
print('In exits but not entries: ', set(ext['Station']) - set(ent['Station']))
print('In entries but not exits: ', set(ent['Station']) - set(ext['Station']))


Missing Station-days 139
By station (top 15):
Station
BTM Layout                               10
Beratena Agrahara                        10
Biocon Hebbagodi                         10
Bommanahalli                             10
Central Silk Board                       10
Delta Electronics Bommasandra            10
Electronic City                          10
Hosa Road                                10
Singasandra                              10
Huskur Road                               9
Hongasandra                               9
Kudlu Gate                                9
Infosys Foundation Konappana Agrahara     8
Jayadeva Hospital                         8
Ragigudda                                 6
dtype: int64
By date (top 10):
Date
2025-08-01    15
2025-08-02    15
2025-08-04    15
2025-08-05    15
2025-08-03    14
2025-08-07    14
2025-08-10    14
2025-08-06    13
2025-08-08    13
2025-08-09    11
dtype: int64
In exits but not entries:  set()
In entries but not exits:  set()


In [8]:
daily = ent.groupby('Date')['Ridership'].sum().reset_index()
daily['dow'] = pd.to_datetime(daily['Date']).dt.day_name()
print('Daily total entries by day of week: ')
print(daily.groupby('dow')['Ridership'].agg(['count', 'mean', 'min', 'max']).round(0))
print('Lowest 5 days:')
print(daily.sort_values('Ridership').head(5).to_string(index=False))

Daily total entries by day of week: 
           count      mean     min     max
dow                                       
Friday         7  698475.0  545543  795987
Monday         8  763921.0  708417  795066
Saturday       7  656177.0  565926  719661
Sunday         7  513968.0  482205  556415
Thursday       6  778815.0  714243  843684
Tuesday        7  757103.0  687984  778020
Wednesday      6  778928.0  732508  800230
Lowest 5 days:
      Date  Ridership    dow
2025-08-10     482205 Sunday
2025-09-21     491549 Sunday
2025-08-03     498746 Sunday
2025-09-07     499672 Sunday
2025-09-28     526641 Sunday


In [9]:
import numpy as np
df = ent.copy()
df["Date"] = pd.to_datetime(df["Date"])
df["dow"] = df["Date"].dt.dayofweek          
df["day_type"] = np.select([df["dow"] < 5, df["dow"] == 5], ["weekday", "saturday"], "sunday")

first_date = df.groupby("Station")["Date"].min()
all_days = pd.to_datetime(sorted(ext["Date"].unique()))
bad = []
for st in df["Station"].unique():
    have_days = set(df.loc[df["Station"] == st, "Date"])
    missing = [d for d in all_days if d not in have_days]
    if missing and max(missing) > first_date[st]:
        bad.append((st, len(missing)))
print("Stations with gaps AFTER their first data day (need a look):", bad if bad else "none")
print("Stations that start late:")
print(first_date[first_date > all_days.min()].sort_values().to_string())

daily = df.groupby(["Date", "dow"])["Ridership"].sum().reset_index()
daily["vs_dow_avg"] = daily["Ridership"] / daily.groupby("dow")["Ridership"].transform("median")
print("\nDays >15% away from their weekday's median:")
print(daily[(daily["vs_dow_avg"] < 0.85) | (daily["vs_dow_avg"] > 1.15)]
      .assign(day=lambda x: x["Date"].dt.day_name()).to_string(index=False))

peak = df.groupby("Station")["Ridership"].quantile(0.95).rename("peak")
df = df.merge(peak, on="Station")
df["crowd"] = (df["Ridership"] / df["peak"]).clip(0, 1)

top = df.groupby("Station")["Ridership"].sum().nlargest(3).index
prof = (df[df["Station"].isin(top)]
        .groupby(["Station", "day_type", "Hour"])["crowd"].mean().round(2)
        .unstack("Hour")[[3, 8, 9, 13, 18, 22]])
print("\nAverage crowd index by station, day type and hour:")
print(prof.to_string())

C:\Users\TANISH N JAY\AppData\Local\Temp\ipykernel_16004\1681687451.py:4: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["dow"] = df["Date"].dt.dayofweek


Stations with gaps AFTER their first data day (need a look): [('Jayadeva Hospital', 8), ('Hongasandra', 9), ('Ragigudda', 6), ('Infosys Foundation Konappana Agrahara', 8), ('Kudlu Gate', 9), ('Huskur Road', 9)]
Stations that start late:
Station
Jayadeva Hospital                       2025-08-03
Hongasandra                             2025-08-06
Ragigudda                               2025-08-06
Infosys Foundation Konappana Agrahara   2025-08-07
Kudlu Gate                              2025-08-08
Huskur Road                             2025-08-09
BTM Layout                              2025-08-11
Beratena Agrahara                       2025-08-11
Biocon Hebbagodi                        2025-08-11
Bommanahalli                            2025-08-11
Central Silk Board                      2025-08-11
Delta Electronics Bommasandra           2025-08-11
Electronic City                         2025-08-11
Hosa Road                               2025-08-11
Singasandra                             2

In [10]:
open_date = pd.Timestamp("2025-08-11")

odd = ["Jayadeva Hospital", "Hongasandra", "Ragigudda",
       "Infosys Foundation Konappana Agrahara", "Kudlu Gate", "Huskur Road"]
early = df[df["Station"].isin(odd) & (df["Date"] < open_date)]
print(early.groupby(["Station", "Date"])["Ridership"].sum().to_string())

clean = df[df["Date"] >= open_date].copy()
n_st, n_days = clean["Station"].nunique(), clean["Date"].nunique()
rows = clean.groupby(["Station", "Date"]).size()
print(f"\nstations={n_st}, days={n_days}, expected station-days={n_st*n_days}, got={len(rows)}")
print("every station-day has 24 hours:", (rows == 24).all())
print("date range:", clean["Date"].min().date(), "to", clean["Date"].max().date())

Station                                Date      
Hongasandra                            2025-08-06     2
Huskur Road                            2025-08-09    11
Infosys Foundation Konappana Agrahara  2025-08-07    14
                                       2025-08-09     2
Jayadeva Hospital                      2025-08-03     1
                                       2025-08-09     2
Kudlu Gate                             2025-08-08     1
Ragigudda                              2025-08-06    18
                                       2025-08-08    34
                                       2025-08-09    95
                                       2025-08-10    28

stations=83, days=38, expected station-days=3154, got=3154
every station-day has 24 hours: True
date range: 2025-08-11 to 2025-09-30


In [12]:
open_date = pd.Timestamp("2025-08-11")

odd = ["Jayadeva Hospital", "Hongasandra", "Ragigudda",
       "Infosys Foundation Konappana Agrahara", "Kudlu Gate", "Huskur Road"]
early = df[df["Station"].isin(odd) & (df["Date"] < open_date)]
print("Early rows for the 6 odd stations (total entries per day):")
print(early.groupby(["Station", "Date"])["Ridership"].sum().to_string())

clean = df[df["Date"] >= open_date].drop(columns=["peak", "crowd"]).copy()
n_st, n_days = clean["Station"].nunique(), clean["Date"].nunique()
rows = clean.groupby(["Station", "Date"]).size()
print(f"\nstations={n_st}, days={n_days}, expected={n_st * n_days}, got={len(rows)}")
print("every station-day has 24 hours:", (rows == 24).all())
print("date range:", clean["Date"].min().date(), "to", clean["Date"].max().date())

daily = clean.groupby(["Date", "day_of_week" if "day_of_week" in clean else "dow"])["Ridership"].sum().reset_index()
daily.columns = ["Date", "dow", "Ridership"]
daily["vs_dow_median"] = daily["Ridership"] / daily.groupby("dow")["Ridership"].transform("median")
print("\nDays more than 12% away from their weekday's median (clean data):")
print(daily[(daily["vs_dow_median"] < 0.88) | (daily["vs_dow_median"] > 1.12)]
      .assign(day=lambda x: x["Date"].dt.day_name()).to_string(index=False))

Early rows for the 6 odd stations (total entries per day):
Station                                Date      
Hongasandra                            2025-08-06     2
Huskur Road                            2025-08-09    11
Infosys Foundation Konappana Agrahara  2025-08-07    14
                                       2025-08-09     2
Jayadeva Hospital                      2025-08-03     1
                                       2025-08-09     2
Kudlu Gate                             2025-08-08     1
Ragigudda                              2025-08-06    18
                                       2025-08-08    34
                                       2025-08-09    95
                                       2025-08-10    28

stations=83, days=38, expected=3154, got=3154
every station-day has 24 hours: True
date range: 2025-08-11 to 2025-09-30

Days more than 12% away from their weekday's median (clean data):
      Date  dow  Ridership  vs_dow_median    day
2025-08-15    4     668554       0.860

In [13]:
holidays = pd.to_datetime(["2025-08-15"])
clean["is_holiday"] = clean["Date"].isin(holidays)

clean = clean[["Date", "Hour", "Station", "Ridership", "dow", "day_type", "is_holiday"]].reset_index(drop=True)
clean.to_parquet("../data/clean_entries.parquet", index=False)

print(clean.shape)
print(clean.head())
print(clean["is_holiday"].value_counts())

(75696, 7)
        Date  Hour    Station  Ridership  dow day_type  is_holiday
0 2025-08-11     0  Attiguppe          0    0  weekday       False
1 2025-08-11     1  Attiguppe          0    0  weekday       False
2 2025-08-11     2  Attiguppe          0    0  weekday       False
3 2025-08-11     3  Attiguppe          0    0  weekday       False
4 2025-08-11     4  Attiguppe          5    0  weekday       False
is_holiday
False    73704
True      1992
Name: count, dtype: int64
